# 03 — MerRec ALS — Kaggle FULL DATA, RAM-safe, checkpoint/resume

Notebook này sửa trực tiếp vấn đề OOM của bản cũ:

- **KHÔNG** `pandas.read_parquet(cf_train)` toàn bộ file.
- **KHÔNG** tạo Python dict cho ~27M items.
- Dùng `Polars LazyFrame + PyArrow batch + disk-backed CSR`.
- Stage 1 tune trên subset đại diện.
- Stage 2 xác nhận Top config trên **FULL TRAIN**.
- Chọn best bằng `NDCG@20 POSITIVE` trên VAL.
- Refit **FULL TRAIN + VAL**.
- TEST chỉ chạy sau khi đã chốt config.
- Mỗi trial/full train có rolling checkpoint; chạy lại sẽ tiếp tục stage đang dở.
- Final artifact dùng cho backend realtime được lưu gọn trong:
  `/kaggle/working/merrec_models/ALS/`

Dataset mong đợi:

`/kaggle/input/.../merrec_retrieval_training/model_data/{cf_train.parquet, train_user_map.parquet, train_item_map.parquet, interactions/...}`


In [1]:
# Cell 1 — Install dependencies only if missing

import sys
import subprocess
import importlib.util

REQUIRED = {
    "polars": "polars>=1.0",
    "pyarrow": "pyarrow>=15",
    "scipy": "scipy>=1.11",
    "pandas": "pandas>=2.0",
    "psutil": "psutil>=5.9",
    "implicit": "implicit==0.7.2",
}

missing = [
    pip_name
    for module_name, pip_name in REQUIRED.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

print("✅ Dependencies ready")


Installing: ['implicit==0.7.2']
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.0 MB/s eta 0:00:00
✅ Dependencies ready


In [2]:
# Cell 2 — Imports + exact Kaggle dataset auto-detection

from pathlib import Path
import os
import gc
import json
import math
import time
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import scipy.sparse as sp
import psutil

from implicit.als import AlternatingLeastSquares
from IPython.display import display

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Dataset auto-detection: search the whole /kaggle/input.
# This works even when Kaggle adds an extra directory layer.
# ------------------------------------------------------------
INPUT_ROOT = Path("/kaggle/input")

cf_matches = list(INPUT_ROOT.rglob("cf_train.parquet"))

print("cf_train.parquet matches:")
for p in cf_matches:
    print("  ", p)

if len(cf_matches) != 1:
    raise RuntimeError(
        f"Expected exactly 1 cf_train.parquet, found {len(cf_matches)}"
    )

CF_TRAIN = cf_matches[0]
MODEL_DATA = CF_TRAIN.parent

USER_MAP_SOURCE = MODEL_DATA / "train_user_map.parquet"
ITEM_MAP_SOURCE = MODEL_DATA / "train_item_map.parquet"
INTERACTIONS_DIR = MODEL_DATA / "interactions"

TRAIN_GLOB = str(INTERACTIONS_DIR / "split=train" / "*.parquet")
VAL_GLOB   = str(INTERACTIONS_DIR / "split=val"   / "*.parquet")
TEST_GLOB  = str(INTERACTIONS_DIR / "split=test"  / "*.parquet")

for p in [USER_MAP_SOURCE, ITEM_MAP_SOURCE]:
    if not p.exists():
        raise FileNotFoundError(p)

print("\nMODEL_DATA =", MODEL_DATA)
print("USER_MAP   =", USER_MAP_SOURCE)
print("ITEM_MAP   =", ITEM_MAP_SOURCE)


cf_train.parquet matches:
   /kaggle/input/deleted-dataset/merrec_retrieval_training/model_data/cf_train.parquet

MODEL_DATA = /kaggle/input/deleted-dataset/merrec_retrieval_training/model_data
USER_MAP   = /kaggle/input/deleted-dataset/merrec_retrieval_training/model_data/train_user_map.parquet
ITEM_MAP   = /kaggle/input/deleted-dataset/merrec_retrieval_training/model_data/train_item_map.parquet


In [3]:
# Cell 3 — Compact output structure + configuration

ROOT = Path("/kaggle/working/merrec_models/ALS")

PROCESSED = ROOT / "processed"
CHECKPOINT = ROOT / "checkpoint"
TUNING = ROOT / "tuning"
MODEL_DIR = ROOT / "model"
MAPPINGS = ROOT / "mappings"
CONFIG_DIR = ROOT / "config"
METRICS = ROOT / "metrics"
STATUS_DIR = ROOT / "status"

for p in [
    ROOT, PROCESSED, CHECKPOINT, TUNING,
    MODEL_DIR, MAPPINGS, CONFIG_DIR, METRICS, STATUS_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- Data preparation ----------
TRAIN_MAPPED = PROCESSED / "train_mapped.parquet"
TRAIN_CSR_DIR = PROCESSED / "train_csr"

# ---------- Stage 1 tuning ----------
SUBSET_DIR = TUNING / "stage1_subset"
STAGE1_DIR = TUNING / "stage1"
STAGE2_DIR = TUNING / "stage2"
for p in [SUBSET_DIR, STAGE1_DIR, STAGE2_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---------- Final train+val ----------
TRAIN_VAL_USER_MAP = MAPPINGS / "train_val_user_map.parquet"
TRAIN_VAL_ITEM_MAP = MAPPINGS / "train_val_item_map.parquet"
VAL_MAPPED = PROCESSED / "val_mapped.parquet"
TRAIN_VAL_MAPPED = PROCESSED / "train_val_mapped.parquet"
TRAIN_VAL_CSR_DIR = PROCESSED / "train_val_csr"

BEST_CONFIG_PATH = CONFIG_DIR / "best_config.json"
VAL_METRICS_PATH = METRICS / "val_metrics.csv"
TEST_METRICS_PATH = METRICS / "test_metrics.csv"
TUNING_RESULTS_PATH = METRICS / "tuning_results.csv"
FINAL_MODEL_PATH = MODEL_DIR / "final_model.npz"
FINAL_CHECKPOINT_PATH = CHECKPOINT / "final_latest_model.npz"
FINAL_STATE_PATH = CHECKPOINT / "final_state.json"
PIPELINE_COMPLETE = STATUS_DIR / "_PIPELINE_COMPLETE.json"

# ============================================================
# Experiment config
# ============================================================
SEED = 42

POSITIVE_EVENTS = {"like", "cart", "offer", "buy_start", "buy_comp"}
SELECTION_MODE = "POSITIVE"
SELECTION_METRIC = "NDCG@20"

K_LIST = [10, 20, 100]

# Stage 1: only coarse tuning — final model remains FULL DATA.
STAGE1_USERS = 120_000
STAGE1_MAX_ITEMS = 2_000_000
STAGE1_VAL_USERS = 1_500
STAGE1_ITERATIONS = 6

STAGE1_CONFIGS = [
    {"name":"f32_r003_a10", "factors":32, "regularization":0.03, "alpha":10.0},
    {"name":"f32_r008_a20", "factors":32, "regularization":0.08, "alpha":20.0},
    {"name":"f32_r015_a40", "factors":32, "regularization":0.15, "alpha":40.0},

    {"name":"f48_r003_a20", "factors":48, "regularization":0.03, "alpha":20.0},
    {"name":"f48_r008_a30", "factors":48, "regularization":0.08, "alpha":30.0},
    {"name":"f48_r015_a40", "factors":48, "regularization":0.15, "alpha":40.0},

    {"name":"f64_r003_a20", "factors":64, "regularization":0.03, "alpha":20.0},
    {"name":"f64_r008_a30", "factors":64, "regularization":0.08, "alpha":30.0},
    {"name":"f64_r015_a40", "factors":64, "regularization":0.15, "alpha":40.0},
]

# Stage 2: confirm the strongest safe configs on FULL TRAIN.
FULL_TOP_N = 2
FULL_ITERATIONS = 12
FULL_CHECKPOINT_EVERY = 2
STAGE2_VAL_USERS = 3_000

# TEST is final only.
FINAL_TEST_USERS = 5_000

# Safety / storage
RAM_SAFE_FRACTION = 0.80
NUM_THREADS = max(1, os.cpu_count() or 1)

# Export portable sparse matrix after final fit.
# Keep True because you asked to retain the matrix for local demo/reuse.
EXPORT_FINAL_MATRIX_NPZ = True

print("ROOT =", ROOT)
print("NUM_THREADS =", NUM_THREADS)


ROOT = /kaggle/working/merrec_models/ALS
NUM_THREADS = 4


In [4]:
# Cell 4 — Helpers: files, schemas, mappings, resources

NUMERIC_DTYPES = {
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}

# Encourage smaller streaming chunks on large joins.
try:
    pl.Config.set_streaming_chunk_size(100_000)
except Exception:
    pass


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)


def valid_parquet(path):
    path = Path(path)

    if not path.exists() or path.stat().st_size <= 0:
        return False

    try:
        pf = pq.ParquetFile(path)
        _ = pf.metadata.num_rows
        return True
    except Exception:
        return False


def parquet_rows(path):
    return (
        int(pq.ParquetFile(path).metadata.num_rows)
        if valid_parquet(path)
        else 0
    )


def lazy_schema(path_or_glob):
    return dict(
        pl.scan_parquet(str(path_or_glob))
        .collect_schema()
    )


def detect_score_col(path):
    s = lazy_schema(path)

    for c in [
        "implicit_score",
        "score",
        "weight",
        "event_weight",
        "rating",
        "strength",
    ]:
        if c in s and s[c] in NUMERIC_DTYPES:
            return c

    numeric = [
        c for c, dtype in s.items()
        if c not in {"user_id", "item_id", "user_idx", "item_idx"}
        and dtype in NUMERIC_DTYPES
    ]

    return numeric[0] if numeric else None


def infer_map_columns(path, entity):
    s = lazy_schema(path)

    id_col = next(
        (c for c in [f"{entity}_id", entity] if c in s),
        None,
    )

    idx_col = next(
        (
            c
            for c in [
                f"{entity}_idx",
                f"{entity}_index",
                f"{entity}_id_idx",
                "idx",
                "index",
            ]
            if c in s
        ),
        None,
    )

    if id_col is None or idx_col is None:
        raise RuntimeError(
            f"Cannot infer {entity} mapping columns from {path}. Schema={s}"
        )

    return id_col, idx_col


def canonical_map(path, entity):
    id_col, idx_col = infer_map_columns(path, entity)

    return (
        pl.scan_parquet(str(path))
        .select([
            pl.col(id_col).cast(pl.Utf8).alias(f"{entity}_id"),
            pl.col(idx_col).cast(pl.UInt32).alias(f"{entity}_idx"),
        ])
    )


def map_size(path, entity):
    row = (
        canonical_map(path, entity)
        .select([
            pl.len().alias("n"),
            pl.col(f"{entity}_idx").min().alias("mn"),
            pl.col(f"{entity}_idx").max().alias("mx"),
            pl.col(f"{entity}_idx").n_unique().alias("nu"),
        ])
        .collect(engine="streaming")
        .row(0, named=True)
    )

    n = int(row["n"])
    mn = int(row["mn"])
    mx = int(row["mx"])
    nu = int(row["nu"])

    if not (mn == 0 and mx == n - 1 and nu == n):
        raise RuntimeError(
            f"{entity} map is not contiguous 0..N-1. Stats={row}"
        )

    return n


def sink_parquet_atomic(lf, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp = path.with_suffix(path.suffix + ".tmp")

    if tmp.exists():
        tmp.unlink()

    lf.sink_parquet(
        str(tmp),
        compression="zstd",
    )

    if not valid_parquet(tmp):
        raise RuntimeError(f"Invalid parquet temp: {tmp}")

    os.replace(tmp, path)


def print_resources():
    vm = psutil.virtual_memory()
    disk = shutil.disk_usage("/kaggle/working")

    print(
        f"RAM: total={vm.total/1024**3:.2f}GB | "
        f"available={vm.available/1024**3:.2f}GB"
    )
    print(
        f"/kaggle/working free={disk.free/1024**3:.2f}GB"
    )


def folder_size_gb(path):
    path = Path(path)
    total = 0

    if path.exists():
        for p in path.rglob("*"):
            if p.is_file():
                total += p.stat().st_size

    return total / (1024**3)


print_resources()


RAM: total=31.35GB | available=30.31GB
/kaggle/working free=19.50GB


In [5]:
# Cell 5 — Preflight: validate the actual MerRec processed dataset

CF_SCHEMA = lazy_schema(CF_TRAIN)
CF_SCORE_COL = detect_score_col(CF_TRAIN)

N_TRAIN_USERS = map_size(USER_MAP_SOURCE, "user")
N_TRAIN_ITEMS = map_size(ITEM_MAP_SOURCE, "item")
N_CF_ROWS = parquet_rows(CF_TRAIN)

print("CF schema:")
print(CF_SCHEMA)

print("\nDetected score column:", CF_SCORE_COL)
print("TRAIN users :", f"{N_TRAIN_USERS:,}")
print("TRAIN items :", f"{N_TRAIN_ITEMS:,}")
print("CF pairs    :", f"{N_CF_ROWS:,}")

for glob, name in [(VAL_GLOB, "VAL"), (TEST_GLOB, "TEST")]:
    s = lazy_schema(glob)
    missing = {"user_id", "item_id"} - set(s)

    if missing:
        raise RuntimeError(f"{name} missing columns: {sorted(missing)}")

print("\n✅ Preflight passed")
print_resources()


CF schema:
{'user_id': String, 'item_id': String, 'implicit_score': Decimal(precision=38, scale=1), 'interactions': Int64, 'views': Int64, 'likes': Int64, 'carts': Int64, 'offers': Int64, 'buy_starts': Int64, 'purchases': Int64, 'last_ts': Datetime(time_unit='us', time_zone=None)}

Detected score column: interactions
TRAIN users : 2,581,378
TRAIN items : 27,328,461
CF pairs    : 109,718,519

✅ Preflight passed
RAM: total=31.35GB | available=29.56GB
/kaggle/working free=19.50GB


In [6]:
# Cell 6 — Build FULL TRAIN mapped parquet with streaming
#
# This replaces the OOM code:
#   train = pd.read_parquet(CF_TRAIN)
#   giant Python user/item dictionaries
#
# If cf_train already contains user_idx/item_idx, no join is needed.

if valid_parquet(TRAIN_MAPPED):
    print("✅ train_mapped.parquet exists -> REUSE")

else:
    print("🚀 Building FULL TRAIN mapped parquet...")

    base = pl.scan_parquet(str(CF_TRAIN))

    raw_expr = (
        pl.col(CF_SCORE_COL).cast(pl.Float32)
        if CF_SCORE_COL is not None
        else pl.lit(1.0).cast(pl.Float32)
    )

    if {"user_idx", "item_idx"}.issubset(CF_SCHEMA):
        print("cf_train already contains continuous indices.")

        mapped = (
            base
            .select([
                pl.col("user_idx").cast(pl.UInt32),
                pl.col("item_idx").cast(pl.UInt32),
                raw_expr.alias("raw_score"),
            ])
        )

    else:
        print("Streaming join cf_train -> train user/item maps.")

        mapped = (
            base
            .select([
                pl.col("user_id").cast(pl.Utf8),
                pl.col("item_id").cast(pl.Utf8),
                raw_expr.alias("raw_score"),
            ])
            .join(
                canonical_map(USER_MAP_SOURCE, "user"),
                on="user_id",
                how="inner",
            )
            .join(
                canonical_map(ITEM_MAP_SOURCE, "item"),
                on="item_id",
                how="inner",
            )
            .select([
                "user_idx",
                "item_idx",
                "raw_score",
            ])
        )

    # Defensive aggregation in case cf_train has duplicate user-item rows.
    mapped = (
        mapped
        .group_by([
            "user_idx",
            "item_idx",
        ])
        .agg(
            pl.col("raw_score")
            .sum()
            .alias("raw_score")
        )
        .with_columns(
            pl.when(pl.col("raw_score") > 0)
            .then((pl.col("raw_score") + 1.0).log())
            .otherwise(0.0)
            .cast(pl.Float32)
            .alias("strength")
        )
        .select([
            pl.col("user_idx").cast(pl.UInt32),
            pl.col("item_idx").cast(pl.UInt32),
            pl.col("raw_score").cast(pl.Float32),
            pl.col("strength").cast(pl.Float32),
        ])
    )

    sink_parquet_atomic(
        mapped,
        TRAIN_MAPPED,
    )

    atomic_json(
        {
            "success": True,
            "rows": parquet_rows(TRAIN_MAPPED),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
        STATUS_DIR / "_TRAIN_MAPPED_COMPLETE.json",
    )

print("TRAIN_MAPPED rows:", f"{parquet_rows(TRAIN_MAPPED):,}")
print_resources()


🚀 Building FULL TRAIN mapped parquet...
Streaming join cf_train -> train user/item maps.
TRAIN_MAPPED rows: 109,718,519
RAM: total=31.35GB | available=22.44GB
/kaggle/working free=18.70GB


In [7]:
# Cell 7 — Build/load disk-backed CSR without COO/Pandas OOM

def csr_complete(matrix_dir):
    matrix_dir = Path(matrix_dir)

    return all(
        (matrix_dir / name).exists()
        for name in [
            "data.f32",
            "indices.i32",
            "indptr.npy",
            "meta.json",
            "_SUCCESS.json",
        ]
    )


def load_disk_csr(matrix_dir):
    matrix_dir = Path(matrix_dir)

    meta = json.loads(
        (matrix_dir / "meta.json")
        .read_text(encoding="utf-8")
    )

    nnz = int(meta["nnz"])
    n_users = int(meta["n_users"])
    n_items = int(meta["n_items"])

    data = np.memmap(
        matrix_dir / "data.f32",
        dtype=np.float32,
        mode="r+",
        shape=(nnz,),
    )

    indices = np.memmap(
        matrix_dir / "indices.i32",
        dtype=np.int32,
        mode="r+",
        shape=(nnz,),
    )

    indptr = np.load(
        matrix_dir / "indptr.npy",
        mmap_mode="r",
    )

    csr = sp.csr_matrix(
        (
            data,
            indices,
            indptr,
        ),
        shape=(
            n_users,
            n_items,
        ),
        copy=False,
    )

    return csr


def build_disk_csr(
    mapped_path,
    matrix_dir,
    n_users,
    n_items,
    batch_size=750_000,
):
    mapped_path = Path(mapped_path)
    matrix_dir = Path(matrix_dir)

    matrix_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if csr_complete(matrix_dir):
        print("✅ CSR checkpoint exists -> REUSE")
        return load_disk_csr(matrix_dir)

    # Clean only incomplete matrix artifacts.
    for name in [
        "data.f32",
        "indices.i32",
        "indptr.npy",
        "meta.json",
        "_SUCCESS.json",
        "counts.i64",
    ]:
        p = matrix_dir / name
        if p.exists():
            p.unlink()

    pf = pq.ParquetFile(mapped_path)
    nnz = int(pf.metadata.num_rows)

    print(
        f"Building CSR: "
        f"shape=({n_users:,}, {n_items:,}), nnz={nnz:,}"
    )

    # --------------------------------------------------------
    # Pass 1: count interactions per user
    # --------------------------------------------------------
    counts_path = matrix_dir / "counts.i64"

    counts = np.memmap(
        counts_path,
        dtype=np.int64,
        mode="w+",
        shape=(n_users,),
    )
    counts[:] = 0

    seen = 0

    for batch in pf.iter_batches(
        batch_size=batch_size,
        columns=["user_idx"],
    ):
        u = np.asarray(
            batch.column(0).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.int32,
        )

        counts[:] += np.bincount(
            u,
            minlength=n_users,
        )

        seen += len(u)

        print(
            f"Pass 1: {seen:,}/{nnz:,}"
        )

    counts.flush()

    indptr = np.empty(
        n_users + 1,
        dtype=np.int64,
    )

    indptr[0] = 0

    np.cumsum(
        np.asarray(counts),
        out=indptr[1:],
    )

    if int(indptr[-1]) != nnz:
        raise RuntimeError(
            f"CSR count mismatch: {indptr[-1]} != {nnz}"
        )

    np.save(
        matrix_dir / "indptr.npy",
        indptr,
    )

    # --------------------------------------------------------
    # Pass 2: scatter item_idx + strength into disk memmaps
    # --------------------------------------------------------
    data = np.memmap(
        matrix_dir / "data.f32",
        dtype=np.float32,
        mode="w+",
        shape=(nnz,),
    )

    indices = np.memmap(
        matrix_dir / "indices.i32",
        dtype=np.int32,
        mode="w+",
        shape=(nnz,),
    )

    cursor = indptr[:-1].copy()

    seen = 0

    for batch in pf.iter_batches(
        batch_size=batch_size,
        columns=[
            "user_idx",
            "item_idx",
            "strength",
        ],
    ):
        u = np.asarray(
            batch.column(0).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.int32,
        )

        i = np.asarray(
            batch.column(1).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.int32,
        )

        v = np.asarray(
            batch.column(2).to_numpy(
                zero_copy_only=False
            ),
            dtype=np.float32,
        )

        order = np.argsort(
            u,
            kind="stable",
        )

        us = u[order]
        is_ = i[order]
        vs = v[order]

        unique_u, starts, lens = np.unique(
            us,
            return_index=True,
            return_counts=True,
        )

        local_offsets = (
            np.arange(
                len(us),
                dtype=np.int64,
            )
            -
            np.repeat(
                starts,
                lens,
            )
        )

        positions = (
            np.repeat(
                cursor[unique_u],
                lens,
            )
            +
            local_offsets
        )

        indices[positions] = is_
        data[positions] = vs

        cursor[unique_u] += lens

        seen += len(u)

        print(
            f"Pass 2: {seen:,}/{nnz:,}"
        )

    data.flush()
    indices.flush()

    if not np.array_equal(
        cursor,
        indptr[1:],
    ):
        raise RuntimeError(
            "CSR cursor mismatch after Pass 2"
        )

    meta = {
        "n_users": int(n_users),
        "n_items": int(n_items),
        "nnz": int(nnz),
        "created_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    }

    atomic_json(
        meta,
        matrix_dir / "meta.json",
    )

    atomic_json(
        {
            "success": True,
            **meta,
        },
        matrix_dir / "_SUCCESS.json",
    )

    del counts
    gc.collect()

    if counts_path.exists():
        counts_path.unlink()

    csr = load_disk_csr(matrix_dir)

    # implicit benefits from sorted CSR rows.
    if not csr.has_sorted_indices:
        print("Sorting CSR item indices...")
        csr.sort_indices()

    return csr


TRAIN_CSR = build_disk_csr(
    TRAIN_MAPPED,
    TRAIN_CSR_DIR,
    N_TRAIN_USERS,
    N_TRAIN_ITEMS,
)

print(
    "TRAIN CSR:",
    TRAIN_CSR.shape,
    "nnz=",
    f"{TRAIN_CSR.nnz:,}",
)

print_resources()


Building CSR: shape=(2,581,378, 27,328,461), nnz=109,718,519
Pass 1: 750,000/109,718,519
Pass 1: 1,500,000/109,718,519
Pass 1: 2,250,000/109,718,519
Pass 1: 3,000,000/109,718,519
Pass 1: 3,750,000/109,718,519
Pass 1: 4,500,000/109,718,519
Pass 1: 5,250,000/109,718,519
Pass 1: 6,000,000/109,718,519
Pass 1: 6,750,000/109,718,519
Pass 1: 7,500,000/109,718,519
Pass 1: 8,250,000/109,718,519
Pass 1: 9,000,000/109,718,519
Pass 1: 9,750,000/109,718,519
Pass 1: 10,500,000/109,718,519
Pass 1: 11,250,000/109,718,519
Pass 1: 12,000,000/109,718,519
Pass 1: 12,750,000/109,718,519
Pass 1: 13,500,000/109,718,519
Pass 1: 14,250,000/109,718,519
Pass 1: 15,000,000/109,718,519
Pass 1: 15,750,000/109,718,519
Pass 1: 16,500,000/109,718,519
Pass 1: 17,250,000/109,718,519
Pass 1: 18,000,000/109,718,519
Pass 1: 18,750,000/109,718,519
Pass 1: 19,500,000/109,718,519
Pass 1: 20,250,000/109,718,519
Pass 1: 21,000,000/109,718,519
Pass 1: 21,750,000/109,718,519
Pass 1: 22,500,000/109,718,519
Pass 1: 23,250,000/109,7

In [8]:
# Cell 8 — Build/load Stage-1 representative subset

SUBSET_MATRIX_PATH = SUBSET_DIR / "subset_matrix.npz"
SUBSET_USER_MAP = SUBSET_DIR / "subset_user_map.parquet"
SUBSET_ITEM_MAP = SUBSET_DIR / "subset_item_map.parquet"

if (
    SUBSET_MATRIX_PATH.exists()
    and valid_parquet(SUBSET_USER_MAP)
    and valid_parquet(SUBSET_ITEM_MAP)
):
    print("✅ Stage-1 subset exists -> REUSE")

    SUBSET_CSR = sp.load_npz(
        SUBSET_MATRIX_PATH
    ).tocsr()

else:
    print("🚀 Building Stage-1 representative subset...")

    rng = np.random.default_rng(SEED)

    # Sample users without loading any user IDs into Python dicts.
    n_sample = min(
        STAGE1_USERS,
        N_TRAIN_USERS,
    )

    full_user_idx = np.sort(
        rng.choice(
            N_TRAIN_USERS,
            size=n_sample,
            replace=False,
        ).astype(np.int32)
    )

    sampled_rows = (
        TRAIN_CSR[
            full_user_idx
        ]
        .tocsr()
    )

    # Keep items actually seen by these users.
    unique_items, counts = np.unique(
        sampled_rows.indices,
        return_counts=True,
    )

    if len(unique_items) > STAGE1_MAX_ITEMS:
        keep = np.argpartition(
            counts,
            -STAGE1_MAX_ITEMS,
        )[
            -STAGE1_MAX_ITEMS:
        ]

        full_item_idx = np.sort(
            unique_items[keep]
            .astype(np.int32)
        )
    else:
        full_item_idx = np.sort(
            unique_items.astype(np.int32)
        )

    SUBSET_CSR = (
        sampled_rows[
            :,
            full_item_idx,
        ]
        .tocsr()
    )

    SUBSET_CSR.sum_duplicates()
    SUBSET_CSR.eliminate_zeros()
    SUBSET_CSR.sort_indices()

    sp.save_npz(
        SUBSET_MATRIX_PATH,
        SUBSET_CSR,
        compressed=True,
    )

    # Create small subset ID maps for validation.
    user_key = pl.DataFrame({
        "full_user_idx": full_user_idx,
        "user_idx": np.arange(
            len(full_user_idx),
            dtype=np.uint32,
        ),
    }).lazy()

    item_key = pl.DataFrame({
        "full_item_idx": full_item_idx,
        "item_idx": np.arange(
            len(full_item_idx),
            dtype=np.uint32,
        ),
    }).lazy()

    subset_users = (
        canonical_map(
            USER_MAP_SOURCE,
            "user",
        )
        .rename({
            "user_idx": "full_user_idx"
        })
        .join(
            user_key,
            on="full_user_idx",
            how="inner",
        )
        .select([
            "user_id",
            "user_idx",
        ])
    )

    subset_items = (
        canonical_map(
            ITEM_MAP_SOURCE,
            "item",
        )
        .rename({
            "item_idx": "full_item_idx"
        })
        .join(
            item_key,
            on="full_item_idx",
            how="inner",
        )
        .select([
            "item_id",
            "item_idx",
        ])
    )

    sink_parquet_atomic(
        subset_users,
        SUBSET_USER_MAP,
    )

    sink_parquet_atomic(
        subset_items,
        SUBSET_ITEM_MAP,
    )

    del sampled_rows
    del unique_items
    del counts
    gc.collect()

print(
    "Stage-1 CSR:",
    SUBSET_CSR.shape,
    "nnz=",
    f"{SUBSET_CSR.nnz:,}",
)

print_resources()


🚀 Building Stage-1 representative subset...
Stage-1 CSR: (120000, 2000000) nnz= 3,254,338
RAM: total=31.35GB | available=29.79GB
/kaggle/working free=17.84GB


In [9]:
# Cell 9 — Evaluation helpers
#
# Evaluation is memory-safe:
# - sample only a few thousand users
# - scan the large maps to resolve only IDs used by that sample
# - cold target items stay in Recall/NDCG denominator
# - coverage is reported separately

def future_lazy(split):
    glob = (
        VAL_GLOB
        if split.lower() == "val"
        else TEST_GLOB
    )

    s = lazy_schema(glob)

    cols = [
        pl.col("user_id").cast(pl.Utf8),
        pl.col("item_id").cast(pl.Utf8),
    ]

    event_col = next(
        (
            c
            for c in [
                "event_group",
                "event",
                "event_type",
            ]
            if c in s
        ),
        None,
    )

    if event_col is not None:
        cols.append(
            pl.col(event_col)
            .cast(pl.Utf8)
            .alias("event_group")
        )

    if "is_strong_positive" in s:
        cols.append(
            pl.col("is_strong_positive")
            .cast(pl.Int8)
        )

    if "is_purchase" in s:
        cols.append(
            pl.col("is_purchase")
            .cast(pl.Int8)
        )

    return (
        pl.scan_parquet(glob)
        .select(cols)
    )


def filter_mode_df(df, mode):
    if mode == "ALL":
        return df

    if mode == "POSITIVE":
        if "event_group" not in df.columns:
            return df.head(0)

        return df.filter(
            pl.col("event_group")
            .is_in(
                sorted(POSITIVE_EVENTS)
            )
        )

    if mode == "STRONG":
        if "is_strong_positive" in df.columns:
            return df.filter(
                pl.col("is_strong_positive") == 1
            )

        return df.head(0)

    if mode == "PURCHASE":
        if "is_purchase" in df.columns:
            return df.filter(
                pl.col("is_purchase") == 1
            )

        if "event_group" in df.columns:
            return df.filter(
                pl.col("event_group")
                == "buy_comp"
            )

        return df.head(0)

    raise ValueError(mode)


def select_eval_cohort(
    split,
    user_map_path,
    max_users,
):
    f = future_lazy(split)

    # Candidate users sorted deterministically by hash.
    candidate_n = max(
        max_users * 20,
        max_users,
    )

    candidate_df = (
        f
        .select("user_id")
        .unique()
        .with_columns(
            pl.col("user_id")
            .hash(seed=SEED)
            .alias("_hash")
        )
        .sort("_hash")
        .head(candidate_n)
        .collect(engine="streaming")
    )

    candidate_ids = (
        candidate_df["user_id"]
        .to_list()
    )

    warm_users = (
        canonical_map(
            user_map_path,
            "user",
        )
        .filter(
            pl.col("user_id")
            .is_in(candidate_ids)
        )
        .collect(engine="streaming")
    )

    cohort = (
        candidate_df
        .join(
            warm_users,
            on="user_id",
            how="inner",
        )
        .sort("_hash")
        .head(max_users)
        .select([
            "user_id",
            "user_idx",
        ])
    )

    if cohort.height == 0:
        raise RuntimeError(
            f"No warm evaluation users found for {split}"
        )

    return cohort


def build_eval_ground_truth(
    split,
    user_map_path,
    item_map_path,
    max_users,
):
    cohort = select_eval_cohort(
        split,
        user_map_path,
        max_users,
    )

    cohort_ids = (
        cohort["user_id"]
        .to_list()
    )

    # Future interactions only for the small evaluation cohort.
    future_df = (
        future_lazy(split)
        .filter(
            pl.col("user_id")
            .is_in(cohort_ids)
        )
        .collect(engine="streaming")
        .join(
            cohort,
            on="user_id",
            how="inner",
        )
    )

    # Scan the huge item map only for target item IDs in this cohort.
    target_item_ids = (
        future_df["item_id"]
        .unique()
        .to_list()
    )

    target_item_map = (
        canonical_map(
            item_map_path,
            "item",
        )
        .filter(
            pl.col("item_id")
            .is_in(target_item_ids)
        )
        .collect(engine="streaming")
    )

    item_idx_lookup = {
        item_id: int(item_idx)
        for item_id, item_idx
        in target_item_map.iter_rows()
    }

    modes = {}

    for mode in [
        "ALL",
        "POSITIVE",
        "STRONG",
        "PURCHASE",
    ]:
        mode_df = (
            filter_mode_df(
                future_df,
                mode,
            )
            .select([
                "user_idx",
                "item_id",
            ])
            .unique()
        )

        if mode_df.height == 0:
            continue

        gt = {}

        for uid, group in mode_df.group_by(
            "user_idx"
        ):
            uid_int = int(
                uid[0]
                if isinstance(uid, tuple)
                else uid
            )

            item_ids = (
                group["item_id"]
                .to_list()
            )

            warm = {
                item_idx_lookup[x]
                for x in item_ids
                if x in item_idx_lookup
            }

            gt[uid_int] = {
                "total_count": len(item_ids),
                "warm_items": warm,
            }

        targets_all = int(
            mode_df.height
        )

        targets_warm = int(
            sum(
                len(info["warm_items"])
                for info in gt.values()
            )
        )

        modes[mode] = {
            "gt": gt,
            "targets_all": targets_all,
            "targets_warm": targets_warm,
            "target_coverage": (
                targets_warm
                / max(
                    targets_all,
                    1,
                )
            ),
        }

    return cohort, modes


def recommend_users(
    model,
    user_items,
    user_ids,
    n,
    batch_size=64,
):
    user_ids = np.asarray(
        user_ids,
        dtype=np.int32,
    )

    predictions = {}

    for start in range(
        0,
        len(user_ids),
        batch_size,
    ):
        ids = user_ids[
            start:start + batch_size
        ]

        histories = user_items[ids]

        try:
            rec_ids, _ = model.recommend(
                ids,
                histories,
                N=n,
                filter_already_liked_items=True,
            )

            if rec_ids.ndim == 1:
                rec_ids = rec_ids[None, :]

            for uid, row in zip(
                ids,
                rec_ids,
            ):
                predictions[int(uid)] = [
                    int(x)
                    for x in row
                    if int(x) >= 0
                ]

        except Exception:
            # Compatibility fallback for versions without batch recommend.
            for uid in ids:
                rec_ids, _ = model.recommend(
                    int(uid),
                    user_items[int(uid)],
                    N=n,
                    filter_already_liked_items=True,
                )

                predictions[int(uid)] = [
                    int(x)
                    for x in rec_ids
                    if int(x) >= 0
                ]

    return predictions


def metrics_at_k(
    predictions,
    gt,
    k,
):
    recalls = []
    precisions = []
    hitrates = []
    ndcgs = []

    for uid, info in gt.items():
        total_count = max(
            int(info["total_count"]),
            1,
        )

        warm_truth = info[
            "warm_items"
        ]

        pred = predictions.get(
            int(uid),
            [],
        )[:k]

        hits = [
            1
            if item in warm_truth
            else 0
            for item in pred
        ]

        n_hit = int(
            sum(hits)
        )

        recalls.append(
            n_hit
            / total_count
        )

        precisions.append(
            n_hit
            / k
        )

        hitrates.append(
            float(
                n_hit > 0
            )
        )

        dcg = sum(
            rel
            / math.log2(rank + 2)
            for rank, rel
            in enumerate(hits)
        )

        ideal_n = min(
            total_count,
            k,
        )

        idcg = sum(
            1.0
            / math.log2(rank + 2)
            for rank in range(
                ideal_n
            )
        )

        ndcgs.append(
            dcg / idcg
            if idcg > 0
            else 0.0
        )

    if not recalls:
        return {
            f"Recall@{k}": np.nan,
            f"Precision@{k}": np.nan,
            f"HitRate@{k}": np.nan,
            f"NDCG@{k}": np.nan,
        }

    return {
        f"Recall@{k}": float(
            np.mean(recalls)
        ),
        f"Precision@{k}": float(
            np.mean(precisions)
        ),
        f"HitRate@{k}": float(
            np.mean(hitrates)
        ),
        f"NDCG@{k}": float(
            np.mean(ndcgs)
        ),
    }


def evaluate_model(
    model,
    user_items,
    split,
    user_map_path,
    item_map_path,
    max_users,
):
    cohort, modes = (
        build_eval_ground_truth(
            split,
            user_map_path,
            item_map_path,
            max_users,
        )
    )

    user_ids = (
        cohort["user_idx"]
        .to_numpy()
        .astype(np.int32)
    )

    predictions = recommend_users(
        model,
        user_items,
        user_ids,
        n=max(K_LIST),
    )

    rows = []

    for mode, info in modes.items():
        row = {
            "split": split.upper(),
            "target_mode": mode,
            "users_eval": len(
                info["gt"]
            ),
            "targets_all": info[
                "targets_all"
            ],
            "targets_warm": info[
                "targets_warm"
            ],
            "target_coverage": info[
                "target_coverage"
            ],
        }

        for k in K_LIST:
            row.update(
                metrics_at_k(
                    predictions,
                    info["gt"],
                    k,
                )
            )

        rows.append(row)

    return pd.DataFrame(rows)


print("✅ Evaluation helpers ready")


✅ Evaluation helpers ready


In [10]:
# Cell 10 — ALS rolling checkpoint/resume helpers

def make_als(
    params,
    iterations,
):
    return AlternatingLeastSquares(
        factors=int(
            params["factors"]
        ),
        regularization=float(
            params["regularization"]
        ),
        alpha=float(
            params["alpha"]
        ),
        iterations=int(
            iterations
        ),
        dtype=np.float32,
        use_native=True,
        use_cg=True,
        calculate_training_loss=False,
        num_threads=NUM_THREADS,
        random_state=SEED,
    )


def atomic_model_save(
    model,
    path,
):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp = path.with_name(
        path.stem
        + ".tmp.npz"
    )

    if tmp.exists():
        tmp.unlink()

    model.save(
        str(tmp)
    )

    os.replace(
        tmp,
        path,
    )


def train_resume(
    params,
    csr,
    checkpoint_path,
    state_path,
    total_iterations,
    chunk_iterations,
):
    checkpoint_path = Path(
        checkpoint_path
    )

    state_path = Path(
        state_path
    )

    checkpoint_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed = 0

    if (
        checkpoint_path.exists()
        and state_path.exists()
    ):
        state = json.loads(
            state_path.read_text(
                encoding="utf-8"
            )
        )

        completed = int(
            state.get(
                "completed_iterations",
                0,
            )
        )

        model = (
            AlternatingLeastSquares
            .load(
                str(checkpoint_path)
            )
        )

        # Ensure current Kaggle CPU thread count.
        model.num_threads = NUM_THREADS

        print(
            f"♻️ RESUME {params['name']}: "
            f"{completed}/{total_iterations} iterations"
        )

    else:
        model = make_als(
            params,
            iterations=1,
        )

        print(
            "🚀 NEW TRAIN:",
            params["name"],
        )

    while completed < total_iterations:
        run_iters = min(
            chunk_iterations,
            total_iterations
            - completed,
        )

        model.iterations = int(
            run_iters
        )

        t0 = time.time()

        model.fit(
            csr,
            show_progress=True,
        )

        elapsed = (
            time.time()
            - t0
        )

        completed += run_iters

        # One rolling checkpoint only.
        atomic_model_save(
            model,
            checkpoint_path,
        )

        atomic_json(
            {
                **params,
                "completed_iterations": completed,
                "target_iterations": total_iterations,
                "last_chunk_minutes": elapsed / 60,
                "updated_at": time.strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            },
            state_path,
        )

        print(
            f"✅ checkpoint "
            f"{completed}/{total_iterations}"
        )

        print_resources()

    return model


def estimate_full_als_gb(
    csr,
    factors,
):
    n_users, n_items = csr.shape

    factor_bytes = (
        (n_users + n_items)
        * int(factors)
        * 4
    )

    # CSR data + indices + indptr.
    csr_bytes = (
        csr.nnz * 8
        +
        (n_users + 1) * 8
    )

    # Conservative working estimate for ALS:
    # latent factors + solver buffers + sparse matrix + Python overhead.
    estimated = (
        factor_bytes * 2.0
        +
        csr_bytes * 1.25
        +
        1.5 * (1024**3)
    )

    return (
        estimated
        / (1024**3)
    )


print("✅ ALS checkpoint helpers ready")


✅ ALS checkpoint helpers ready


In [11]:
# Cell 11 — Stage 1 hyperparameter screening on subset
#
# Every completed trial has metrics.json.
# Re-running the notebook skips completed trials.

stage1_records = []

for params in STAGE1_CONFIGS:
    trial_dir = (
        STAGE1_DIR
        / params["name"]
    )

    trial_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        trial_dir
        / "metrics.json"
    )

    checkpoint_path = (
        trial_dir
        / "latest_model.npz"
    )

    state_path = (
        trial_dir
        / "state.json"
    )

    if metrics_path.exists():
        print(
            "✅ Stage1 REUSE:",
            params["name"],
        )

        stage1_records.append(
            json.loads(
                metrics_path.read_text(
                    encoding="utf-8"
                )
            )
        )

        continue

    model = train_resume(
        params=params,
        csr=SUBSET_CSR,
        checkpoint_path=checkpoint_path,
        state_path=state_path,
        total_iterations=STAGE1_ITERATIONS,
        chunk_iterations=2,
    )

    result = evaluate_model(
        model=model,
        user_items=SUBSET_CSR,
        split="val",
        user_map_path=SUBSET_USER_MAP,
        item_map_path=SUBSET_ITEM_MAP,
        max_users=STAGE1_VAL_USERS,
    )

    display(result)

    selected = result[
        result["target_mode"]
        == SELECTION_MODE
    ]

    score = (
        float(
            selected.iloc[0][
                SELECTION_METRIC
            ]
        )
        if not selected.empty
        else float("-inf")
    )

    record = {
        **params,
        "stage": "stage1_subset",
        "status": "complete",
        "iterations": STAGE1_ITERATIONS,
        "selection_metric": SELECTION_METRIC,
        "selection_value": score,
        "metrics": result.to_dict(
            orient="records"
        ),
    }

    atomic_json(
        record,
        metrics_path,
    )

    stage1_records.append(
        record
    )

    # Metrics are enough after a Stage-1 trial.
    # Delete its factors to save Kaggle output space.
    for p in [
        checkpoint_path,
        state_path,
    ]:
        if p.exists():
            p.unlink()

    del model
    gc.collect()


stage1_ranked = sorted(
    [
        x
        for x in stage1_records
        if x.get("status")
        == "complete"
    ],
    key=lambda x: float(
        x["selection_value"]
    ),
    reverse=True,
)

if not stage1_ranked:
    raise RuntimeError(
        "No valid Stage-1 ALS trial."
    )

atomic_json(
    {
        "success": True,
        "ranked": stage1_ranked,
    },
    STAGE1_DIR
    / "_STAGE1_COMPLETE.json",
)

stage1_table = pd.DataFrame([
    {
        "stage": "stage1_subset",
        "name": x["name"],
        "factors": x["factors"],
        "regularization": x[
            "regularization"
        ],
        "alpha": x["alpha"],
        "selection_metric": (
            SELECTION_METRIC
        ),
        "selection_value": x[
            "selection_value"
        ],
    }
    for x in stage1_ranked
])

display(
    stage1_table
)

print("✅ Stage 1 complete")


🚀 NEW TRAIN: f32_r003_a10


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 2/6
RAM: total=31.35GB | available=29.62GB
/kaggle/working free=17.59GB


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 4/6
RAM: total=31.35GB | available=29.62GB
/kaggle/working free=17.59GB


...

  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 6/6
RAM: total=31.35GB | available=29.34GB
/kaggle/working free=17.34GB


,split,target_mode,users_eval,targets_all,targets_warm,target_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,Recall@20,Precision@20,HitRate@20,NDCG@20,Recall@100,Precision@100,HitRate@100,NDCG@100
0,VAL,ALL,1207,19773,4455,0.225307,0.000346,0.000829,0.0058,0.000867,0.000543,0.000704,0.010771,0.000903,0.003134,0.000472,0.030655,0.001611
1,VAL,POSITIVE,559,2725,607,0.222752,0.000000,0.000000,0.0000,0.000000,0.000596,0.000089,0.001789,0.000205,0.001904,0.000072,0.007156,0.000592
2,VAL,STRONG,203,430,139,0.323256,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,VAL,PURCHASE,7,10,3,0.300000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,stage,name,factors,regularization,alpha,selection_metric,selection_value
0,stage1_subset,f48_r003_a20,48,0.03,20.0,NDCG@20,0.000575
1,stage1_subset,f32_r015_a40,32,0.15,40.0,NDCG@20,0.000530
2,stage1_subset,f32_r008_a20,32,0.08,20.0,NDCG@20,0.000526
3,stage1_subset,f64_r003_a20,64,0.03,20.0,NDCG@20,0.000470
4,stage1_subset,f32_r003_a10,32,0.03,10.0,NDCG@20,0.000467
5,stage1_subset,f48_r008_a30,48,0.08,30.0,NDCG@20,0.000299
6,stage1_subset,f48_r015_a40,48,0.15,40.0,NDCG@20,0.000280
7,stage1_subset,f64_r008_a30,64,0.08,30.0,NDCG@20,0.000227
8,stage1_subset,f64_r015_a40,64,0.15,40.0,NDCG@20,0.000205


✅ Stage 1 complete


In [12]:
# Cell 12 — Select RAM-safe configs for FULL TRAIN confirmation

stage1_ranked = json.loads(
    (
        STAGE1_DIR
        / "_STAGE1_COMPLETE.json"
    ).read_text(
        encoding="utf-8"
    )
)["ranked"]

vm = psutil.virtual_memory()

safe_budget_gb = (
    vm.total
    / (1024**3)
    * RAM_SAFE_FRACTION
)

FULL_CONFIGS = []

print(
    "RAM safe budget:",
    f"{safe_budget_gb:.2f} GB",
)

for x in stage1_ranked:
    params = {
        "name": x["name"],
        "factors": int(
            x["factors"]
        ),
        "regularization": float(
            x["regularization"]
        ),
        "alpha": float(
            x["alpha"]
        ),
    }

    estimate = estimate_full_als_gb(
        TRAIN_CSR,
        params["factors"],
    )

    safe = (
        estimate
        <= safe_budget_gb
    )

    print(
        params["name"],
        "| estimate=",
        f"{estimate:.2f}GB",
        "|",
        "SAFE ✅"
        if safe
        else "SKIP ⚠️",
    )

    if safe:
        FULL_CONFIGS.append(
            params
        )

    if (
        len(FULL_CONFIGS)
        >= FULL_TOP_N
    ):
        break


if not FULL_CONFIGS:
    lightest = min(
        stage1_ranked,
        key=lambda x: int(
            x["factors"]
        ),
    )

    estimate = estimate_full_als_gb(
        TRAIN_CSR,
        int(
            lightest["factors"]
        ),
    )

    raise MemoryError(
        "Kaggle RAM hiện tại không đủ an toàn "
        "để FULL ALS. "
        f"Config nhẹ nhất ước tính ~{estimate:.2f}GB. "
        "Checkpoint preprocessing vẫn còn, "
        "không phải làm lại."
    )


print("\nFULL CONFIGS:")
for x in FULL_CONFIGS:
    print(x)


RAM safe budget: 25.08 GB
f48_r003_a20 | estimate= 13.24GB | SAFE ✅
f32_r015_a40 | estimate= 9.68GB | SAFE ✅

FULL CONFIGS:
{'name': 'f48_r003_a20', 'factors': 48, 'regularization': 0.03, 'alpha': 20.0}
{'name': 'f32_r015_a40', 'factors': 32, 'regularization': 0.15, 'alpha': 40.0}


In [13]:
# Cell 13 — Stage 2: confirm Top configs on FULL TRAIN + evaluate VAL
#
# Only trial metrics are retained after completion.
# The massive trial factors are deleted after metrics are saved.

stage2_records = []

for params in FULL_CONFIGS:
    trial_dir = (
        STAGE2_DIR
        / params["name"]
    )

    trial_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        trial_dir
        / "metrics.json"
    )

    checkpoint_path = (
        trial_dir
        / "latest_model.npz"
    )

    state_path = (
        trial_dir
        / "state.json"
    )

    if metrics_path.exists():
        print(
            "✅ Stage2 REUSE:",
            params["name"],
        )

        stage2_records.append(
            json.loads(
                metrics_path.read_text(
                    encoding="utf-8"
                )
            )
        )

        continue

    model = train_resume(
        params=params,
        csr=TRAIN_CSR,
        checkpoint_path=checkpoint_path,
        state_path=state_path,
        total_iterations=FULL_ITERATIONS,
        chunk_iterations=FULL_CHECKPOINT_EVERY,
    )

    val_result = evaluate_model(
        model=model,
        user_items=TRAIN_CSR,
        split="val",
        user_map_path=USER_MAP_SOURCE,
        item_map_path=ITEM_MAP_SOURCE,
        max_users=STAGE2_VAL_USERS,
    )

    display(
        val_result
    )

    selected = val_result[
        val_result["target_mode"]
        == SELECTION_MODE
    ]

    score = (
        float(
            selected.iloc[0][
                SELECTION_METRIC
            ]
        )
        if not selected.empty
        else float("-inf")
    )

    record = {
        **params,
        "stage": "stage2_full_train",
        "status": "complete",
        "iterations": FULL_ITERATIONS,
        "selection_metric": (
            SELECTION_METRIC
        ),
        "selection_value": score,
        "metrics": val_result.to_dict(
            orient="records"
        ),
    }

    atomic_json(
        record,
        metrics_path,
    )

    stage2_records.append(
        record
    )

    # Full trial factor matrices can be several GB.
    # Metrics are safely stored, so remove factors before next trial.
    for p in [
        checkpoint_path,
        state_path,
    ]:
        if p.exists():
            p.unlink()

    del model
    gc.collect()


valid_stage2 = [
    x
    for x in stage2_records
    if x.get("status")
    == "complete"
]

if not valid_stage2:
    raise RuntimeError(
        "No FULL TRAIN Stage-2 trial completed."
    )

best = max(
    valid_stage2,
    key=lambda x: float(
        x["selection_value"]
    ),
)

BEST_CONFIG = {
    "name": best["name"],
    "factors": int(
        best["factors"]
    ),
    "regularization": float(
        best["regularization"]
    ),
    "alpha": float(
        best["alpha"]
    ),
    "iterations": int(
        best["iterations"]
    ),
    "selection_mode": (
        SELECTION_MODE
    ),
    "selection_metric": (
        SELECTION_METRIC
    ),
    "selection_value": float(
        best["selection_value"]
    ),
}

atomic_json(
    BEST_CONFIG,
    BEST_CONFIG_PATH,
)

# Save VAL metrics of the actual FULL TRAIN best config.
pd.DataFrame(
    best["metrics"]
).to_csv(
    VAL_METRICS_PATH,
    index=False,
)

# Unified tuning CSV.
stage2_table = pd.DataFrame([
    {
        "stage": "stage2_full_train",
        "name": x["name"],
        "factors": x["factors"],
        "regularization": x[
            "regularization"
        ],
        "alpha": x["alpha"],
        "selection_metric": (
            SELECTION_METRIC
        ),
        "selection_value": x[
            "selection_value"
        ],
    }
    for x in valid_stage2
])

pd.concat(
    [
        stage1_table,
        stage2_table,
    ],
    ignore_index=True,
).to_csv(
    TUNING_RESULTS_PATH,
    index=False,
)

print("=" * 72)
print("BEST FULL ALS CONFIG")
print("=" * 72)
print(
    json.dumps(
        BEST_CONFIG,
        ensure_ascii=False,
        indent=2,
    )
)

print("\nVAL METRICS")
display(
    pd.read_csv(
        VAL_METRICS_PATH
    )
)


🚀 NEW TRAIN: f48_r003_a20


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 2/12
RAM: total=31.35GB | available=24.44GB
/kaggle/working free=12.49GB


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 4/12
RAM: total=31.35GB | available=24.47GB
/kaggle/working free=12.49GB


...

  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 12/12
RAM: total=31.35GB | available=26.14GB
/kaggle/working free=14.28GB


,split,target_mode,users_eval,targets_all,targets_warm,target_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,Recall@20,Precision@20,HitRate@20,NDCG@20,Recall@100,Precision@100,HitRate@100,NDCG@100
0,VAL,ALL,3000,44821,33319,0.743379,0.000495,0.000700,0.006000,0.000859,0.001625,0.000633,0.011000,0.001070,0.004862,0.000483,0.031667,0.001902
1,VAL,POSITIVE,1367,6556,4959,0.756406,0.000100,0.000146,0.001463,0.000163,0.000100,0.000073,0.001463,0.000127,0.001718,0.000095,0.008778,0.000455
2,VAL,STRONG,517,1085,848,0.781567,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000967,0.000019,0.001934,0.000191
3,VAL,PURCHASE,34,50,39,0.780000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


BEST FULL ALS CONFIG
{
  "name": "f32_r015_a40",
  "factors": 32,
  "regularization": 0.15,
  "alpha": 40.0,
  "iterations": 12,
  "selection_mode": "POSITIVE",
  "selection_metric": "NDCG@20",
  "selection_value": 0.00012723593376680995
}

VAL METRICS


,split,target_mode,users_eval,targets_all,targets_warm,target_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,Recall@20,Precision@20,HitRate@20,NDCG@20,Recall@100,Precision@100,HitRate@100,NDCG@100
0,VAL,ALL,3000,44821,33319,0.743379,0.000495,0.000700,0.006000,0.000859,0.001625,0.000633,0.011000,0.001070,0.004862,0.000483,0.031667,0.001902
1,VAL,POSITIVE,1367,6556,4959,0.756406,0.000100,0.000146,0.001463,0.000163,0.000100,0.000073,0.001463,0.000127,0.001718,0.000095,0.008778,0.000455
2,VAL,STRONG,517,1085,848,0.781567,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000967,0.000019,0.001934,0.000191
3,VAL,PURCHASE,34,50,39,0.780000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [14]:
# Cell 14 — Expand maps with VAL-only users/items
#
# TRAIN indices stay unchanged.
# New VAL-only IDs are appended after the TRAIN universe.

def build_expanded_map(
    source_train_map,
    entity,
    output_path,
):
    output_path = Path(
        output_path
    )

    if valid_parquet(
        output_path
    ):
        print(
            "✅ expanded map REUSE:",
            output_path,
        )
        return

    train_lf = canonical_map(
        source_train_map,
        entity,
    )

    n_train = map_size(
        source_train_map,
        entity,
    )

    f = future_lazy("val")

    new_ids = (
        f
        .select(
            f"{entity}_id"
        )
        .unique()
        .join(
            train_lf.select(
                f"{entity}_id"
            ),
            on=f"{entity}_id",
            how="anti",
        )
        .sort(
            f"{entity}_id"
        )
        .with_row_index(
            f"{entity}_idx",
            offset=n_train,
        )
        .with_columns(
            pl.col(
                f"{entity}_idx"
            ).cast(pl.UInt32)
        )
        .select([
            f"{entity}_id",
            f"{entity}_idx",
        ])
    )

    combined = pl.concat(
        [
            train_lf.select([
                f"{entity}_id",
                f"{entity}_idx",
            ]),
            new_ids,
        ],
        how="vertical_relaxed",
    )

    sink_parquet_atomic(
        combined,
        output_path,
    )


build_expanded_map(
    USER_MAP_SOURCE,
    "user",
    TRAIN_VAL_USER_MAP,
)

build_expanded_map(
    ITEM_MAP_SOURCE,
    "item",
    TRAIN_VAL_ITEM_MAP,
)

N_TRAINVAL_USERS = map_size(
    TRAIN_VAL_USER_MAP,
    "user",
)

N_TRAINVAL_ITEMS = map_size(
    TRAIN_VAL_ITEM_MAP,
    "item",
)

print(
    "TRAIN+VAL users:",
    f"{N_TRAINVAL_USERS:,}",
)

print(
    "TRAIN+VAL items:",
    f"{N_TRAINVAL_ITEMS:,}",
)


TRAIN+VAL users: 2,693,525
TRAIN+VAL items: 28,964,787


In [15]:
# Cell 15 — Build VAL mapped data + FULL TRAIN+VAL mapped parquet

def event_weight_expr(s):
    if "event_weight" in s:
        return (
            pl.col("event_weight")
            .cast(pl.Float32)
            .alias("raw_score")
        )

    event_col = next(
        (
            c
            for c in [
                "event_group",
                "event",
                "event_type",
            ]
            if c in s
        ),
        None,
    )

    if event_col is None:
        return (
            pl.lit(1.0)
            .cast(pl.Float32)
            .alias("raw_score")
        )

    return (
        pl.when(
            pl.col(event_col)
            == "buy_comp"
        ).then(10.0)
        .when(
            pl.col(event_col)
            == "buy_start"
        ).then(7.0)
        .when(
            pl.col(event_col)
            == "offer"
        ).then(5.0)
        .when(
            pl.col(event_col)
            == "cart"
        ).then(4.0)
        .when(
            pl.col(event_col)
            == "like"
        ).then(2.0)
        .otherwise(1.0)
        .cast(pl.Float32)
        .alias("raw_score")
    )


if valid_parquet(VAL_MAPPED):
    print("✅ VAL_MAPPED REUSE")

else:
    print("🚀 Mapping VAL interactions...")

    val_schema = lazy_schema(
        VAL_GLOB
    )

    val_lf = (
        pl.scan_parquet(
            VAL_GLOB
        )
        .select([
            pl.col("user_id")
            .cast(pl.Utf8),
            pl.col("item_id")
            .cast(pl.Utf8),
            event_weight_expr(
                val_schema
            ),
        ])
        .group_by([
            "user_id",
            "item_id",
        ])
        .agg(
            pl.col("raw_score")
            .sum()
            .alias("raw_score")
        )
        .join(
            canonical_map(
                TRAIN_VAL_USER_MAP,
                "user",
            ),
            on="user_id",
            how="inner",
        )
        .join(
            canonical_map(
                TRAIN_VAL_ITEM_MAP,
                "item",
            ),
            on="item_id",
            how="inner",
        )
        .select([
            pl.col("user_idx")
            .cast(pl.UInt32),
            pl.col("item_idx")
            .cast(pl.UInt32),
            pl.col("raw_score")
            .cast(pl.Float32),
        ])
    )

    sink_parquet_atomic(
        val_lf,
        VAL_MAPPED,
    )


if valid_parquet(TRAIN_VAL_MAPPED):
    print("✅ TRAIN_VAL_MAPPED REUSE")

else:
    print("🚀 Merge FULL TRAIN + VAL...")

    merged = (
        pl.concat(
            [
                pl.scan_parquet(
                    str(TRAIN_MAPPED)
                )
                .select([
                    "user_idx",
                    "item_idx",
                    "raw_score",
                ]),

                pl.scan_parquet(
                    str(VAL_MAPPED)
                )
                .select([
                    "user_idx",
                    "item_idx",
                    "raw_score",
                ]),
            ],
            how="vertical_relaxed",
        )
        .group_by([
            "user_idx",
            "item_idx",
        ])
        .agg(
            pl.col("raw_score")
            .sum()
            .alias("raw_score")
        )
        .with_columns(
            pl.when(
                pl.col("raw_score") > 0
            )
            .then(
                (
                    pl.col("raw_score")
                    + 1.0
                ).log()
            )
            .otherwise(0.0)
            .cast(pl.Float32)
            .alias("strength")
        )
        .select([
            pl.col("user_idx")
            .cast(pl.UInt32),
            pl.col("item_idx")
            .cast(pl.UInt32),
            pl.col("raw_score")
            .cast(pl.Float32),
            pl.col("strength")
            .cast(pl.Float32),
        ])
    )

    sink_parquet_atomic(
        merged,
        TRAIN_VAL_MAPPED,
    )


print(
    "VAL mapped:",
    f"{parquet_rows(VAL_MAPPED):,}",
)

print(
    "TRAIN+VAL mapped:",
    f"{parquet_rows(TRAIN_VAL_MAPPED):,}",
)


🚀 Mapping VAL interactions...
🚀 Merge FULL TRAIN + VAL...
VAL mapped: 13,632,771
TRAIN+VAL mapped: 121,779,472


In [16]:
# Cell 16 — Build/load FULL TRAIN+VAL disk-backed CSR

TRAIN_VAL_CSR = build_disk_csr(
    TRAIN_VAL_MAPPED,
    TRAIN_VAL_CSR_DIR,
    N_TRAINVAL_USERS,
    N_TRAINVAL_ITEMS,
)

print(
    "TRAIN+VAL CSR:",
    TRAIN_VAL_CSR.shape,
    "nnz=",
    f"{TRAIN_VAL_CSR.nnz:,}",
)

print_resources()


Building CSR: shape=(2,693,525, 28,964,787), nnz=121,779,472
Pass 1: 750,000/121,779,472
Pass 1: 1,500,000/121,779,472
Pass 1: 2,250,000/121,779,472
Pass 1: 3,000,000/121,779,472
Pass 1: 3,750,000/121,779,472
Pass 1: 4,500,000/121,779,472
Pass 1: 5,250,000/121,779,472
Pass 1: 6,000,000/121,779,472
Pass 1: 6,750,000/121,779,472
Pass 1: 7,500,000/121,779,472
Pass 1: 8,250,000/121,779,472
Pass 1: 9,000,000/121,779,472
Pass 1: 9,750,000/121,779,472
Pass 1: 10,500,000/121,779,472
Pass 1: 11,250,000/121,779,472
Pass 1: 12,000,000/121,779,472
Pass 1: 12,750,000/121,779,472
Pass 1: 13,500,000/121,779,472
Pass 1: 14,250,000/121,779,472
Pass 1: 15,000,000/121,779,472
Pass 1: 15,750,000/121,779,472
Pass 1: 16,500,000/121,779,472
Pass 1: 17,250,000/121,779,472
Pass 1: 18,000,000/121,779,472
Pass 1: 18,750,000/121,779,472
Pass 1: 19,500,000/121,779,472
Pass 1: 20,250,000/121,779,472
Pass 1: 21,000,000/121,779,472
Pass 1: 21,750,000/121,779,472
Pass 1: 22,500,000/121,779,472
Pass 1: 23,250,000/121,7

In [17]:
# Cell 17 — FINAL FULL TRAIN+VAL fit with rolling checkpoint/resume

BEST_CONFIG = json.loads(
    BEST_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

if FINAL_MODEL_PATH.exists():
    print("✅ FINAL MODEL already exists -> load")

    FINAL_MODEL = (
        AlternatingLeastSquares
        .load(
            str(FINAL_MODEL_PATH)
        )
    )

else:
    estimate = estimate_full_als_gb(
        TRAIN_VAL_CSR,
        BEST_CONFIG[
            "factors"
        ],
    )

    total_ram_gb = (
        psutil.virtual_memory().total
        / (1024**3)
    )

    print(
        "Final ALS memory estimate:",
        f"{estimate:.2f} GB",
    )

    print(
        "Kaggle total RAM:",
        f"{total_ram_gb:.2f} GB",
    )

    if (
        estimate
        >
        total_ram_gb
        * RAM_SAFE_FRACTION
    ):
        raise MemoryError(
            "Final ALS bị chặn trước model.fit để tránh OOM. "
            "Các preprocessing/tuning checkpoint đã lưu đầy đủ."
        )

    FINAL_MODEL = train_resume(
        params=BEST_CONFIG,
        csr=TRAIN_VAL_CSR,
        checkpoint_path=FINAL_CHECKPOINT_PATH,
        state_path=FINAL_STATE_PATH,
        total_iterations=int(
            BEST_CONFIG[
                "iterations"
            ]
        ),
        chunk_iterations=FULL_CHECKPOINT_EVERY,
    )

    # Move the final rolling checkpoint instead of duplicating
    # several GB of latent factors.
    if FINAL_MODEL_PATH.exists():
        FINAL_MODEL_PATH.unlink()

    shutil.move(
        str(FINAL_CHECKPOINT_PATH),
        str(FINAL_MODEL_PATH),
    )

    if FINAL_STATE_PATH.exists():
        FINAL_STATE_PATH.unlink()

    print(
        "✅ FINAL MODEL saved:",
        FINAL_MODEL_PATH,
    )


print(
    "User factors:",
    FINAL_MODEL.user_factors.shape,
)

print(
    "Item factors:",
    FINAL_MODEL.item_factors.shape,
)

print_resources()


Final ALS memory estimate: 10.21 GB
Kaggle total RAM: 31.35 GB
🚀 NEW TRAIN: f32_r015_a40


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 2/12
RAM: total=31.35GB | available=25.98GB
/kaggle/working free=11.88GB


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 4/12
RAM: total=31.35GB | available=25.96GB
/kaggle/working free=11.88GB


...

✅ checkpoint 8/12
RAM: total=31.35GB | available=26.02GB
/kaggle/working free=11.88GB


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 10/12
RAM: total=31.35GB | available=26.02GB
/kaggle/working free=11.88GB


  0%|          | 0/2 [00:00<?, ?it/s]

✅ checkpoint 12/12
RAM: total=31.35GB | available=26.03GB
/kaggle/working free=11.88GB
✅ FINAL MODEL saved: /kaggle/working/merrec_models/ALS/model/final_model.npz
User factors: (2693525, 32)
Item factors: (28964787, 32)
RAM: total=31.35GB | available=26.03GB
/kaggle/working free=11.88GB


In [18]:
# Cell 18 — FINAL TEST evaluation
#
# TEST is not used anywhere in tuning/config selection.

if TEST_METRICS_PATH.exists():
    print("✅ TEST metrics already exist -> REUSE")

    TEST_RESULT = pd.read_csv(
        TEST_METRICS_PATH
    )

else:
    TEST_RESULT = evaluate_model(
        model=FINAL_MODEL,
        user_items=TRAIN_VAL_CSR,
        split="test",
        user_map_path=TRAIN_VAL_USER_MAP,
        item_map_path=TRAIN_VAL_ITEM_MAP,
        max_users=FINAL_TEST_USERS,
    )

    TEST_RESULT.to_csv(
        TEST_METRICS_PATH,
        index=False,
    )

display(
    TEST_RESULT
)

print("✅ Final TEST evaluation complete")


,split,target_mode,users_eval,targets_all,targets_warm,target_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,Recall@20,Precision@20,HitRate@20,NDCG@20,Recall@100,Precision@100,HitRate@100,NDCG@100
0,TEST,ALL,5000,63940,48925,0.765170,0.000454,0.00058,0.0056,0.000717,0.000755,0.000490,0.008400,0.000725,0.002948,0.000336,0.025200,0.001354
1,TEST,POSITIVE,2201,10023,7639,0.762147,0.000000,0.00000,0.0000,0.000000,0.001122,0.000114,0.002272,0.000347,0.002532,0.000073,0.007269,0.000689
2,TEST,STRONG,733,1496,1211,0.809492,0.000000,0.00000,0.0000,0.000000,0.001364,0.000068,0.001364,0.000369,0.001364,0.000014,0.001364,0.000369
3,TEST,PURCHASE,32,38,35,0.921053,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


✅ Final TEST evaluation complete


In [19]:
# Cell 19 — Export deployable artifacts for the realtime web backend

# Final mappings used by the final TRAIN+VAL model are already stored in MAPPINGS/.
# Export a portable CSR NPZ only once if requested.

FINAL_MATRIX_NPZ = (
    PROCESSED
    / "train_val_matrix.npz"
)

if EXPORT_FINAL_MATRIX_NPZ:
    if FINAL_MATRIX_NPZ.exists():
        print(
            "✅ Portable matrix already exists:",
            FINAL_MATRIX_NPZ,
        )
    else:
        print(
            "Exporting portable train_val_matrix.npz..."
        )

        sp.save_npz(
            FINAL_MATRIX_NPZ,
            TRAIN_VAL_CSR,
            compressed=True,
        )

        print(
            "✅ saved:",
            FINAL_MATRIX_NPZ,
        )


# Artifact manifest for backend/deployment.
manifest = {
    "model_name": "ALS",
    "model_type": "implicit_collaborative_filtering",
    "version": "v1",
    "model_path": str(
        FINAL_MODEL_PATH
    ),
    "user_map_path": str(
        TRAIN_VAL_USER_MAP
    ),
    "item_map_path": str(
        TRAIN_VAL_ITEM_MAP
    ),
    "matrix_path": (
        str(FINAL_MATRIX_NPZ)
        if FINAL_MATRIX_NPZ.exists()
        else None
    ),
    "best_config_path": str(
        BEST_CONFIG_PATH
    ),
    "val_metrics_path": str(
        VAL_METRICS_PATH
    ),
    "test_metrics_path": str(
        TEST_METRICS_PATH
    ),
    "trained_on": "TRAIN+VAL",
    "selection": (
        f"{SELECTION_METRIC} {SELECTION_MODE} on VAL"
    ),
    "created_at": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}

atomic_json(
    manifest,
    ROOT / "artifact_manifest.json",
)

atomic_json(
    {
        "success": True,
        "best_config": BEST_CONFIG,
        "output_root": str(ROOT),
        "completed_at": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        ),
    },
    PIPELINE_COMPLETE,
)

print(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nALS output size:",
    f"{folder_size_gb(ROOT):.2f} GB",
)


Exporting portable train_val_matrix.npz...
✅ saved: /kaggle/working/merrec_models/ALS/processed/train_val_matrix.npz
{
  "model_name": "ALS",
  "model_type": "implicit_collaborative_filtering",
  "version": "v1",
  "model_path": "/kaggle/working/merrec_models/ALS/model/final_model.npz",
  "user_map_path": "/kaggle/working/merrec_models/ALS/mappings/train_val_user_map.parquet",
  "item_map_path": "/kaggle/working/merrec_models/ALS/mappings/train_val_item_map.parquet",
  "matrix_path": "/kaggle/working/merrec_models/ALS/processed/train_val_matrix.npz",
  "best_config_path": "/kaggle/working/merrec_models/ALS/config/best_config.json",
  "val_metrics_path": "/kaggle/working/merrec_models/ALS/metrics/val_metrics.csv",
  "test_metrics_path": "/kaggle/working/merrec_models/ALS/metrics/test_metrics.csv",
  "trained_on": "TRAIN+VAL",
  "selection": "NDCG@20 POSITIVE on VAL",
  "created_at": "2026-09-21 17:50:25"
}

ALS output size: 8.06 GB


In [20]:
# Cell 20 — Create serving ZIP for download to your local web demo
#
# The ZIP intentionally excludes the huge sparse matrix.
# The matrix remains available separately in:
#   processed/train_val_matrix.npz
#
# Realtime serving needs:
#   final_model.npz
#   train_val_user_map.parquet
#   train_val_item_map.parquet
#   best_config.json
#   metrics
#   artifact_manifest.json

SERVING_ZIP = Path(
    "/kaggle/working/ALS_serving_package.zip"
)

if SERVING_ZIP.exists():
    SERVING_ZIP.unlink()

include_files = [
    FINAL_MODEL_PATH,
    TRAIN_VAL_USER_MAP,
    TRAIN_VAL_ITEM_MAP,
    BEST_CONFIG_PATH,
    VAL_METRICS_PATH,
    TEST_METRICS_PATH,
    TUNING_RESULTS_PATH,
    ROOT / "artifact_manifest.json",
    PIPELINE_COMPLETE,
]

with zipfile.ZipFile(
    SERVING_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    allowZip64=True,
) as zf:

    for p in include_files:
        p = Path(p)

        if not p.exists():
            print(
                "⚠️ missing, skip:",
                p,
            )
            continue

        arcname = p.relative_to(
            ROOT
        )

        print(
            "ADD:",
            arcname,
        )

        zf.write(
            p,
            arcname=str(arcname),
        )

print(
    "\n✅ DOWNLOAD PACKAGE:",
    SERVING_ZIP,
)

print(
    "ZIP size:",
    f"{SERVING_ZIP.stat().st_size/1024**3:.2f} GB",
)


ADD: model/final_model.npz
ADD: mappings/train_val_user_map.parquet
ADD: mappings/train_val_item_map.parquet
ADD: config/best_config.json
ADD: metrics/val_metrics.csv
ADD: metrics/test_metrics.csv
ADD: metrics/tuning_results.csv
ADD: artifact_manifest.json
ADD: status/_PIPELINE_COMPLETE.json

✅ DOWNLOAD PACKAGE: /kaggle/working/ALS_serving_package.zip
ZIP size: 3.75 GB


In [21]:
# Cell 21 — Final status / resume audit

print("=" * 80)
print("MERREC ALS PIPELINE STATUS")
print("=" * 80)

checks = {
    "train_mapped": TRAIN_MAPPED,
    "train_csr": (
        TRAIN_CSR_DIR
        / "_SUCCESS.json"
    ),
    "stage1_complete": (
        STAGE1_DIR
        / "_STAGE1_COMPLETE.json"
    ),
    "best_config": BEST_CONFIG_PATH,
    "val_metrics": VAL_METRICS_PATH,
    "train_val_user_map": (
        TRAIN_VAL_USER_MAP
    ),
    "train_val_item_map": (
        TRAIN_VAL_ITEM_MAP
    ),
    "train_val_mapped": (
        TRAIN_VAL_MAPPED
    ),
    "train_val_csr": (
        TRAIN_VAL_CSR_DIR
        / "_SUCCESS.json"
    ),
    "final_model": (
        FINAL_MODEL_PATH
    ),
    "test_metrics": (
        TEST_METRICS_PATH
    ),
    "pipeline_complete": (
        PIPELINE_COMPLETE
    ),
}

for name, path in checks.items():
    print(
        f"{name:24s}",
        "✅"
        if Path(path).exists()
        else "❌",
        path,
    )

print()

if BEST_CONFIG_PATH.exists():
    print("BEST CONFIG")
    print(
        BEST_CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

print()

if TEST_METRICS_PATH.exists():
    print("FINAL TEST")
    display(
        pd.read_csv(
            TEST_METRICS_PATH
        )
    )

print_resources()

print(
    "\nTotal ALS output:",
    f"{folder_size_gb(ROOT):.2f} GB",
)

print(
    "\nServing ZIP:",
    "/kaggle/working/ALS_serving_package.zip"
)


MERREC ALS PIPELINE STATUS
train_mapped             ✅ /kaggle/working/merrec_models/ALS/processed/train_mapped.parquet
train_csr                ✅ /kaggle/working/merrec_models/ALS/processed/train_csr/_SUCCESS.json
stage1_complete          ✅ /kaggle/working/merrec_models/ALS/tuning/stage1/_STAGE1_COMPLETE.json
best_config              ✅ /kaggle/working/merrec_models/ALS/config/best_config.json
val_metrics              ✅ /kaggle/working/merrec_models/ALS/metrics/val_metrics.csv
train_val_user_map       ✅ /kaggle/working/merrec_models/ALS/mappings/train_val_user_map.parquet
train_val_item_map       ✅ /kaggle/working/merrec_models/ALS/mappings/train_val_item_map.parquet
train_val_mapped         ✅ /kaggle/working/merrec_models/ALS/processed/train_val_mapped.parquet
train_val_csr            ✅ /kaggle/working/merrec_models/ALS/processed/train_val_csr/_SUCCESS.json
final_model              ✅ /kaggle/working/merrec_models/ALS/model/final_model.npz
test_metrics             ✅ /kaggle/working/merr

,split,target_mode,users_eval,targets_all,targets_warm,target_coverage,Recall@10,Precision@10,HitRate@10,NDCG@10,Recall@20,Precision@20,HitRate@20,NDCG@20,Recall@100,Precision@100,HitRate@100,NDCG@100
0,TEST,ALL,5000,63940,48925,0.765170,0.000454,0.00058,0.0056,0.000717,0.000755,0.000490,0.008400,0.000725,0.002948,0.000336,0.025200,0.001354
1,TEST,POSITIVE,2201,10023,7639,0.762147,0.000000,0.00000,0.0000,0.000000,0.001122,0.000114,0.002272,0.000347,0.002532,0.000073,0.007269,0.000689
2,TEST,STRONG,733,1496,1211,0.809492,0.000000,0.00000,0.0000,0.000000,0.001364,0.000068,0.001364,0.000369,0.001364,0.000014,0.001364,0.000369
3,TEST,PURCHASE,32,38,35,0.921053,0.000000,0.00000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


RAM: total=31.35GB | available=25.98GB
/kaggle/working free=7.70GB

Total ALS output: 8.06 GB

Serving ZIP: /kaggle/working/ALS_serving_package.zip
